# EDA — Siniestros en Ruta Chile 2024
## EFT SCY1101 — Programación para la Ciencia de Datos

---

## Qué es este notebook

Este notebook es la etapa de **exploración de datos (EDA)** — previa al ETL formal. Aquí se toman y documentan las decisiones de diseño (qué variables usar, cuáles descartar, cómo se define la variable objetivo) que luego se implementan como código productivo en `/etl/`. **No es el pipeline final**, es el paso de reconocimiento y justificación técnica.


## 1. Setup — librerías y carga de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

# Ruta del CSV (relativa a la raiz del proyecto)
CSV_PATH = "../data/Rural_2024.csv"

df = pd.read_csv(CSV_PATH)
print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.shape

ModuleNotFoundError: No module named 'matplotlib'

## 2. Overview general

Revisión inicial: tipos de datos, nulos, duplicados y primeras filas.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
# Estadísticas descriptivas de columnas numéricas
df.describe().round(2)

In [ ]:
# Valores nulos por columna
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({'nulos': nulos, 'pct': nulos_pct})
print(resumen_nulos[resumen_nulos['nulos'] > 0] if resumen_nulos['nulos'].sum() > 0 else "No hay valores nulos.")

In [ ]:
# Filas duplicadas
dup = df.duplicated().sum()
print(f"Filas duplicadas: {dup} ({dup/len(df)*100:.2f}%)")

## 3. Análisis de variables categóricas

Revisar distribución de `Tipo__CONA`, `Causa__CON`, `Zona`, `Región`/`Comuna` para entender qué representan y decidir cuáles son válidas como *features* del modelo (sin data leakage).

In [ ]:
for col in ["Tipo__CONA", "Causa__CON", "Zona"]:
    print(f"=== {col} ===")
    print(df[col].value_counts())
    print()

In [ ]:
# Gráfico de barras horizontales para las principales categóricas
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Tipo siniestro
tipo_counts = df["Tipo__CONA"].value_counts()
axes[0].barh(tipo_counts.index[::-1], tipo_counts.values[::-1], color="steelblue")
axes[0].set_title("Distribución por Tipo de Siniestro", fontsize=13)
axes[0].set_xlabel("N° de siniestros")
for i, v in enumerate(tipo_counts.values[::-1]):
    axes[0].text(v + 30, i, f"{v:,}", va='center', fontsize=9)

# Causa
causa_counts = df["Causa__CON"].value_counts()
axes[1].barh(causa_counts.index[::-1], causa_counts.values[::-1], color="salmon")
axes[1].set_title("Distribución por Causa (informativa, no es feature)", fontsize=13)
axes[1].set_xlabel("N° de siniestros")
for i, v in enumerate(causa_counts.values[::-1]):
    axes[1].text(v + 30, i, f"{v:,}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 regiones con más siniestros
region_counts = df["REGION_DPA"].value_counts().head(10)
print(region_counts)

region_counts.sort_values().plot(
    kind="barh", figsize=(10, 5),
    title="Top 10 regiones con más siniestros en ruta (2024)",
    color="mediumseagreen"
)
plt.xlabel("N° de siniestros")
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 comunas con más siniestros
top_comunas = df["COMUNA_DPA"].value_counts().head(15)
top_comunas.sort_values().plot(
    kind="barh", figsize=(10, 6),
    title="Top 15 comunas con más siniestros en ruta (2024)",
    color="cornflowerblue"
)
plt.xlabel("N° de siniestros")
plt.tight_layout()
plt.show()

**Nota de diseño:**

- `Zona`: si es constante (100% 'RURAL'), se descarta como feature (varianza cero) — pero informa el alcance real del dataset (siniestros en rutas interurbanas, no urbanos).
- `Causa__CON`: determinación investigativa posterior al siniestro (parte policial) → **no usar como feature de entrada al modelo predictivo**, sí usar en el dashboard como variable descriptiva.
- `Tipo__CONA`: describe el hecho ocurrido. Se evalúa si se conoce *antes* o *junto con* el resultado. Se excluye para evitar leakage, ya que es concurrente con la gravedad.
- `Región`/`Comuna` vs `REGION_DPA`/`COMUNA_DPA`: son redundantes, se usa la versión DPA (codificación oficial).

## 4. Análisis temporal

Distribución de siniestros por mes, día de semana y hora aproximada. Buscar patrones (ej. más siniestros de noche, fines de semana, meses de vacaciones).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
mes_counts = df["Mes"].value_counts().sort_index()
axes[0].bar(mes_counts.index, mes_counts.values, color="steelblue")
axes[0].set_title("Siniestros por mes")
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(meses, rotation=45)

dias = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
dia_counts = df["Diasemana"].value_counts().sort_index()
axes[1].bar(dia_counts.index, dia_counts.values, color="coral")
axes[1].set_title("Siniestros por día de semana")
axes[1].set_xticks(range(1, 8))
axes[1].set_xticklabels(dias[:len(dia_counts)], rotation=45)

hora_counts = df["Hora_aprox"].value_counts().sort_index()
axes[2].bar(hora_counts.index, hora_counts.values, color="mediumpurple")
axes[2].set_title("Siniestros por hora aproximada")
axes[2].set_xlabel("Hora del día")

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: día de semana vs hora del día
pivot_tiempo = df.pivot_table(
    index="Diasemana",
    columns="Hora_aprox",
    values="IdAccident",
    aggfunc="count",
    fill_value=0
)
plt.figure(figsize=(18, 4))
sns.heatmap(pivot_tiempo, cmap="YlOrRd", linewidths=0.3, annot=False)
plt.title("Concentración de siniestros: día de semana vs hora")
plt.xlabel("Hora aproximada")
plt.ylabel("Día de semana (1=Lun ... 7=Dom)")
plt.tight_layout()
plt.show()

In [ ]:
# Feature derivada: fin de semana (si Diasemana=1 es lunes, 6=sáb, 7=dom)
df["es_fin_de_semana"] = df["Diasemana"].isin([6, 7]).astype(int)
print("Distribución fin de semana:")
print(df["es_fin_de_semana"].value_counts().rename({0: 'Semana', 1: 'Fin de semana'}))

# Feature derivada: franja horaria
def franja_horaria(h):
    if 0 <= h < 6: return "Madrugada (0-5)"
    elif 6 <= h < 12: return "Mañana (6-11)"
    elif 12 <= h < 18: return "Tarde (12-17)"
    else: return "Noche (18-23)"

df["franja_horaria"] = df["Hora_aprox"].apply(franja_horaria)
print("\nDistribución por franja horaria:")
print(df["franja_horaria"].value_counts())

## 5. Construcción y balance de la variable objetivo

La gravedad del siniestro se deriva de las columnas `Fallecidos`, `Graves`, `Menos_Grav`, `Leves`.

In [ ]:
# Resumen de columnas de víctimas
cols_victimas = ["Fallecidos", "Graves", "Menos_Grav", "Leves", "Lesionados"]
df[cols_victimas].describe().round(2)

In [ ]:
def clasificar_gravedad(row):
    if row["Fallecidos"] > 0:
        return "Fatal"
    elif row["Graves"] > 0:
        return "Grave"
    elif row["Menos_Grav"] > 0 or row["Leves"] > 0:
        return "Leve"
    else:
        return "Sin lesionados"

df["gravedad"] = df.apply(clasificar_gravedad, axis=1)
conteo = df["gravedad"].value_counts()
print(conteo)
print(f"\nProporción:")
print((conteo / len(df) * 100).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Conteo absoluto
orden = ["Fatal", "Grave", "Leve", "Sin lesionados"]
colores = ["#c0392b", "#e67e22", "#f1c40f", "#27ae60"]
vals = [df["gravedad"].value_counts().get(g, 0) for g in orden]
axes[0].bar(orden, vals, color=colores)
axes[0].set_title("Distribución de la variable objetivo (absoluto)")
axes[0].set_ylabel("N° siniestros")
for i, v in enumerate(vals):
    axes[0].text(i, v + 50, f"{v:,}", ha='center', fontsize=10)

# Pie chart proporcional
axes[1].pie(vals, labels=orden, colors=colores, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Proporción de clases (gravedad)")

plt.tight_layout()
plt.show()

**Nota de diseño:** Si hay desbalance de clases significativo (esperado: pocas filas 'Fatal' vs muchas 'Leve'), se debe documentar la estrategia de modelado:
- `class_weight='balanced'` en Random Forest / Regresión Logística.
- Métricas robustas: F1-macro, AUC-ROC multiclase (OvR).
- Opcionalmente: oversampling con SMOTE en el pipeline de entrenamiento.

## 6. Análisis geográfico

Revisión de la dispersión de coordenadas (`Lat`, `Lon`) — validación de rango de valores y visualización de la distribución espacial de siniestros. Esta información es la que luego se cruza con la API de hospitales (Overpass) para calcular la variable `distancia_hospital_mas_cercano`.

In [ ]:
print("Rango de coordenadas:")
df[["Lat", "Lon"]].describe().round(4)

In [ ]:
# Validar que las coordenadas estén dentro del territorio de Chile
lat_valida = df["Lat"].between(-56, -17)
lon_valida = df["Lon"].between(-76, -66)
invalidos = (~lat_valida | ~lon_valida).sum()
print(f"Registros con coordenadas fuera del rango esperado para Chile: {invalidos}")
print(f"Registros válidos: {(lat_valida & lon_valida).sum():,}")

In [ ]:
# Scatter geográfico coloreado por gravedad
fig, axes = plt.subplots(1, 2, figsize=(14, 10))

# Distribución general
axes[0].scatter(df["Lon"], df["Lat"], s=2, alpha=0.25, color="steelblue")
axes[0].set_title("Distribución geográfica\n(todos los siniestros)")
axes[0].set_xlabel("Longitud")
axes[0].set_ylabel("Latitud")

# Coloreado por gravedad
color_map = {"Fatal": "#c0392b", "Grave": "#e67e22", "Leve": "#f1c40f", "Sin lesionados": "#27ae60"}
for grav, color in color_map.items():
    sub = df[df["gravedad"] == grav]
    axes[1].scatter(sub["Lon"], sub["Lat"], s=2, alpha=0.3, color=color, label=grav)

axes[1].set_title("Distribución geográfica\npor gravedad")
axes[1].set_xlabel("Longitud")
axes[1].set_ylabel("Latitud")
axes[1].legend(markerscale=5, title="Gravedad")

plt.tight_layout()
plt.show()

## 7. Gravedad por región y variables temporales

Análisis cruzado de la variable objetivo con las features candidatas: ¿en qué regiones/horarios hay más siniestros fatales?

In [ ]:
# Porcentaje de siniestros fatales por región (top 10 regiones)
top_regiones = df["REGION_DPA"].value_counts().head(10).index
df_top = df[df["REGION_DPA"].isin(top_regiones)]

tabla = pd.crosstab(
    df_top["REGION_DPA"],
    df_top["gravedad"],
    normalize="index"
).round(3) * 100

# Reordenar columnas
cols_orden = [c for c in ["Fatal", "Grave", "Leve", "Sin lesionados"] if c in tabla.columns]
tabla = tabla[cols_orden]
tabla.sort_values("Fatal", ascending=False)

In [ ]:
# Gravedad por franja horaria
tabla_hora = pd.crosstab(
    df["franja_horaria"],
    df["gravedad"],
    normalize="index"
).round(3) * 100

cols_orden = [c for c in ["Fatal", "Grave", "Leve", "Sin lesionados"] if c in tabla_hora.columns]
tabla_hora[cols_orden].plot(
    kind="bar", stacked=True, figsize=(10, 5),
    color=["#c0392b", "#e67e22", "#f1c40f", "#27ae60"],
    title="% de gravedad por franja horaria"
)
plt.ylabel("%")
plt.xticks(rotation=15)
plt.legend(title="Gravedad", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

## 8. Conclusiones y justificación de decisiones

*(Actualizar según resultados observados en las secciones anteriores)*

### Columnas descartadas y por qué
| Columna | Razón |
|---|---|
| `X`, `Y` | Redundantes con `Lat`/`Lon` |
| `FID`, `IdAccident` | Identificadores sin valor predictivo |
| `Zona` | Varianza cero (100% RURAL) |
| `Región`, `Ciudad`, `Comuna` | Redundantes con versión DPA (codificación oficial) |
| `Causa__CON` | Data leakage: determinación posterior al siniestro |
| `Tipo__CONA` | Data leakage: concurrente con la gravedad |
| `Fallecidos`, `Graves`, `Menos_Grav`, `Leves`, `Lesionados` | Directamente derivados de la variable objetivo |

### Features finales propuestas para el modelo
| Feature | Tipo | Descripción |
|---|---|---|
| `Mes` | Numérica | Mes del año (1–12) |
| `Diasemana` | Numérica | Día de la semana (1–7) |
| `es_fin_de_semana` | Binaria (derivada) | 1 si Sáb o Dom |
| `Hora_aprox` | Numérica | Hora del día (0–23) |
| `franja_horaria` | Categórica (derivada) | Madrugada / Mañana / Tarde / Noche |
| `REGION_DPA` | Categórica | Código región oficial |
| `COMUNA_DPA` | Categórica | Código comuna oficial |
| `Lat`, `Lon` | Numéricas | Coordenadas (para calcular distancia al hospital) |
| `distancia_hospital_mas_cercano` | Numérica (a construir en ETL) | Km al hospital más cercano (via API Overpass) |

### Variable objetivo
- **`gravedad`** (Fatal / Grave / Leve / Sin lesionados) — clasificación multiclase ordenada.

### Estrategia ante desbalance de clases
- Usar `class_weight='balanced'` en todos los modelos sklearn.
- Evaluar con F1-macro y AUC-ROC multiclase (OvR), no con accuracy.
- Analizar SMOTE si la clase 'Fatal' queda con representación < 5%.

---
Estas decisiones alimentan directamente el diseño de `/etl/transform.py` y la justificación técnica del informe ejecutivo.